In [ ]:
!pip install transformers datasets
!pip install tqdm
!pip install peft

In [ ]:
import os
import transformers
import argparse
import torch
import logging
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from peft import PeftModel
from tqdm import tqdm
import sys
import numpy as np
import random
from typing import List, Dict
import math
import random


from transformers import (
    T5Config,
    T5ForConditionalGeneration,
    AutoTokenizer,
    LlamaForCausalLM,
    DataCollatorWithPadding
)


In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/utils/utils.py

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = False



def random_initialization(model, tokenizer, backbone):
    ids = []
    for x in range(30000):
        tokenized_ids = tokenizer.encode(str(x))
        if 3 in tokenized_ids:
            tokenized_ids.remove(3)
        if 1 in tokenized_ids:
            tokenized_ids.remove(1)
        ids += tokenized_ids
    ids = list(set(ids))

    # reinitialize the embedding in the backbone models
    for index in ids:
        if 't5' in backbone:
            model.shared.weight.data[index] = nn.init.normal_(
                model.shared.weight.data[index], 0, 1.0
            )
        elif 'llama' in backbone.lower():
            model.model.embed_tokens.weight.data[index] = nn.init.normal_(
                model.model.embed_tokens.weight.data[index], 0, 1.0
            )

    return model

def setup_logging(
    log_name: str,
    datasets: str,
    log_dir: str,
    model_dir: str,
    checkpoint_dir: str,
    logging_level: int,
    sample_prompt: str,
    his_prefix: str,
    skip_empty_his: int,
    max_his: int,
    master_port: int,
    tasks: str,
    backbone: str,
    item_indexing: str,
    lr: float,
    epochs: int,
    batch_size: int,
    sample_num: int,
    prompt_file: str
):
    log_name = log_name(
        datasets,
        sample_prompt,
        his_prefix,
        skip_empty_his,
        max_his,
        master_port,
        tasks,
        backbone,
        item_indexing,
        lr,
        epochs,
        batch_size,
        sample_num,
        prompt_file
    )
    if len(datasets.split(',')) > 1:
        folder_name = 'SP5'
    else:
        folder_name = datasets
    folder = os.path.join(log_dir, folder_name)
    if not os.path.exists(folder):
        os.makedirs(folder)
    log_file = os.path.join(log_dir, folder_name, log_name + '.log')

    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    logging.basicConfig(filename=log_file, level=logging_level, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))

    return

def log_name(
    datasets,
    sample_prompt,
    his_prefix,
    skip_empty_his,
    max_his,
    master_port,
    tasks,
    backbone,
    item_indexing,
    lr,
    epochs,
    batch_size,
    sample_num,
    prompt_file,
):
    if len(datasets.split(',')) > 1:
        folder_name = 'SP5'
    else:
        folder_name = datasets
    params = [str(sample_prompt), str(his_prefix), str(skip_empty_his), str(max_his), str(master_port), folder_name, tasks, backbone, item_indexing, str(lr), str(epochs), str(batch_size), sample_num, prompt_file[3:-4]]
    return '_'.join(params)

def ReadLineFromFile(path):
    if not os.path.exists(path):
        raise FileNotFoundError
    lines = []
    with open(path,'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

def WriteDictToFile(path, write_dict):
    with open(path, 'w') as out:
        for user, items in write_dict.items():
            if type(items) == list:
                out.write(user + ' ' + ' '.join(items) + '\n')
            else:
                out.write(user + ' ' + str(items) + '\n')


def load_prompt_template(path, task_list):
    """
    Load prompt template from the file. Keep training tasks only.
    Input:
    - path: The path for prompt template txt file.
    - task_list: A list of required tasks.
    Return:
    - prompt_templates: a dictionary of prompt templates. e.g., {task: {'seen': {'0': {'Input': template_input, 'Output': template_output}}}}

    """

    if not os.path.exists(path):
        raise FileNotFoundError
    prompt_info = ReadLineFromFile(path)
    prompt_templates = dict()
    for prompt in prompt_info:
        t = [sens.strip() for sens in prompt.split(';')]
        if t[0] not in task_list:
            continue
        if t[0] not in prompt_templates:
            prompt_templates[t[0]] = dict()
        if t[1] not in prompt_templates[t[0]]:
            prompt_templates[t[0]][t[1]] = dict()
        num = len(prompt_templates[t[0]][t[1]])
        prompt_templates[t[0]][t[1]][str(num)] = dict()
        prompt_templates[t[0]][t[1]][str(num)]['Input'] = t[2]
        prompt_templates[t[0]][t[1]][str(num)]['Output'] = t[3]
    return prompt_templates

def get_info_from_prompt(prompt_templates):
    """
    Extract the require information from the prompt templates.
    Input:
    - prompt_templates: a dictionary of prompt templates.
    Output:
    - info: a list of required information.
    """

    info = []
    for task in prompt_templates:
        for see in prompt_templates[task]:
            for i in prompt_templates[task][see]:
                info += re.findall(r'\{.*?\}', prompt_templates[task][see][i]['Input'])
                info += re.findall(r'\{.*?\}', prompt_templates[task][see][i]['Output'])
    info = [i[1:-1] for i in set(info)]
    return info

def check_task_prompt(prompt_templates, task_list):
    """
    Check if all tasks have prompt templates. Raise Error if training tasks have no prompt.
    Input:
    - prompt_templates: A dictionary of prompt templates.
    - task_list: A list of training tasks.
    """
    for task in task_list:
        assert task in prompt_templates, f"No prompt for {task} task"

In [ ]:
class EvaluationDataset(Dataset):
    def __init__(self, dataset, tokenizer, cutoff):
        super().__init__()
        self.input = tokenizer(
            dataset['input'], padding="longest", truncation=True, max_length=cutoff
        )
        self.output = tokenizer(
            dataset['output'], padding="longest", truncation=True, max_length=cutoff
        )

    def __len__(self):
        return len(self.input["input_ids"])

    def __getitem__(self, index):
        return {
            "input_ids": torch.tensor(self.input["input_ids"][index]),
            "attention_mask": torch.tensor(self.input["attention_mask"][index]),
            'label': torch.tensor(self.output["input_ids"][index])
        }


In [ ]:
# https://github.com/agiresearch/OpenP5/blob/main/src/utils/evaluate.py
def rel_results_filtered(user_positive, id2user, user_idx, return_num, predictions, targets, scores, k):
    results = []
    batch_length = len(targets)
    for b in range(batch_length):
        uidx = user_idx[b]
        user_id = id2user[uidx]
        positive = user_positive[user_id]
        one_batch_sequence = predictions[
            b * return_num : (b + 1) * return_num
        ]
        one_batch_score = scores[
            b * return_num : (b + 1) * return_num
        ]
        pairs = [(a, b) for a, b in zip(one_batch_sequence, one_batch_score)]
        sorted_pairs = sorted(pairs, key=lambda x: x[1], reverse=True)
        gt = targets[b]
        one_results = []
        for sorted_pred in sorted_pairs:
            if sorted_pred[0] not in positive:
                if sorted_pred[0] == gt:
                    one_results.append(1)
                else:
                    one_results.append(0)
                if len(one_results) >= k:
                    break
            else:
                continue

        results.append(one_results)
    return results

def rel_results(predictions, targets, scores, k):
    results = []
    batch_length = len(targets)
    for b in range(batch_length):
        one_batch_sequence = predictions[
            b * k : (b + 1) * k
        ]
        one_batch_score = scores[
            b * k : (b + 1) * k
        ]
        pairs = [(a, b) for a, b in zip(one_batch_sequence, one_batch_score)]
        sorted_pairs = sorted(pairs, key=lambda x: x[1], reverse=True)
        gt = targets[b]
        one_results = []
        for sorted_pred in sorted_pairs:
            if sorted_pred[0] == gt:
                one_results.append(1)
            else:
                one_results.append(0)

        results.append(one_results)
    return results

def get_metrics_results(rel_results, metrics):
    res = []
    for m in metrics:
        if m.lower().startswith('hit'):
            k = int(m.split('@')[1])
            res.append(hit_at_k(rel_results, k))
        elif m.lower().startswith('ndcg'):
            k = int(m.split('@')[1])
            res.append(ndcg_at_k(rel_results, k))

    return np.array(res)

def ndcg_at_k(relevance, k):
    """
    Since we apply leave-one-out, each user only have one ground truth item, so the idcg would be 1.0
    """
    ndcg = 0.0
    for row in relevance:
        rel = row[:k]
        one_ndcg = 0.0
        for i in range(len(rel)):
            one_ndcg += rel[i] / math.log(i+2,2)
        ndcg += one_ndcg
    return ndcg


def hit_at_k(relevance, k):
    correct = 0.0
    for row in relevance:
        rel = row[:k]
        if sum(rel) > 0:
            correct += 1
    return correct

In [ ]:
def test_single_task(model, tokenizer, dataloader, metrics, generate_num, prefix_allowed_tokens):
    test_total = 0
    metrics_res = np.array([0.0] * len(metrics))
    model.cuda()
    for batch in tqdm(dataloader):
        prediction = model.generate(
            input_ids=batch["input_ids"].cuda(),
            attention_mask=batch["attention_mask"].cuda(),
            max_length=30,
            prefix_allowed_tokens_fn=prefix_allowed_tokens,
            num_beams=generate_num,
            num_return_sequences=generate_num,
            output_scores=True,
            return_dict_in_generate=True,
        )

        output_ids = batch['label'].cuda()
        prediction_ids = prediction["sequences"]
        prediction_scores = prediction["sequences_scores"]

        gold_sents = tokenizer.batch_decode(
            output_ids, skip_special_tokens=True
        )
        generated_sents = tokenizer.batch_decode(
            prediction_ids, skip_special_tokens=True
        )

        # print(generated_sents)
        # exit()
        rel_results_ = rel_results(generated_sents, gold_sents, prediction_scores, generate_num)

        test_total += len(rel_results_)

        metrics_res += get_metrics_results(rel_results_, metrics)

    metrics_res = torch.tensor(metrics_res)
    test_total = torch.tensor(test_total)

    metrics_res /= test_total

    for i in range(len(metrics)):
        logging.info(f'{metrics[i]}: {metrics_res[i]}')



In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/utils/indexing.py

def sequential_indexing(data_path, dataset, user_sequence_dict, order):
    """
    Use sequential indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, f'item_sequential_indexing_{order}.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_sequential_indexing_{order}.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = dict()
        if order == 'original':
            user_list = user_sequence_dict.keys()
        elif order == 'short2long':
            user_list = sorted(user_sequence_dict, key=lambda x: len(user_sequence_dict[x]), reverse=False)
        elif order == 'long2short':
            user_list = sorted(user_sequence_dict, key=lambda x: len(user_sequence_dict[x]), reverse=True)

        for user in user_list:
            items = user_sequence_dict[user][:-2]
            for item in items:
                if item not in item_map:
                    item_map[item] = str(len(item_map) + 1001)
        for user in user_list:
            items = user_sequence_dict[user][-2:]
            for item in items:
                if item not in item_map:
                    item_map[item] = str(len(item_map) + 1001)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map



def random_indexing(data_path, dataset, user_sequence_dict):
    """
    Use random indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, 'item_random_indexing.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_random_indexing.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = dict()
        items = set()
        for user in user_sequence_dict:
            items.update(user_sequence_dict[user])
        items = list(items)
        random.shuffle(items)
        for item in items:
            if item not in item_map:
                item_map[item] = str(len(item_map) + 1001)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map

def collaborative_indexing(data_path, dataset, user_sequence_dict, token_size, cluster_num, last_token, float32):
    """
    Use collaborative indexing method to index the given user seuqnece dict.
    """
    user_index_file = os.path.join(data_path, dataset, 'user_indexing.txt')
    item_index_file = os.path.join(data_path, dataset, f'item_collaborative_indexing_{token_size}_{cluster_num}_{last_token}.txt')
    reindex_sequence_file = os.path.join(data_path, dataset, f'user_sequence_collaborative_indexing_{token_size}_{cluster_num}_{last_token}.txt')

    if os.path.exists(reindex_sequence_file):
        user_sequence = ReadLineFromFile(reindex_sequence_file)

        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)

        return construct_user_sequence_dict(user_sequence), item_map

    # For user index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(user_index_file):
        user_info = ReadLineFromFile(user_index_file)
        user_map = get_dict_from_lines(user_info)
    else:
        user_map = generate_user_map(user_sequence_dict)
        WriteDictToFile(user_index_file, user_map)


    # For item index, load from txt file if already exists, otherwise generate from user sequence and save.
    if os.path.exists(item_index_file):
        item_info = ReadLineFromFile(item_index_file)
        item_map = get_dict_from_lines(item_info)
    else:
        item_map = generate_collaborative_id(user_sequence_dict, token_size, cluster_num, last_token, float32)
        WriteDictToFile(item_index_file, item_map)

    reindex_user_sequence_dict = reindex(user_sequence_dict, user_map, item_map)
    WriteDictToFile(reindex_sequence_file, reindex_user_sequence_dict)
    return reindex_user_sequence_dict, item_map

def generate_collaborative_id(user_sequence_dict, token_size, cluster_num, last_token, float32):
    """
    Generate collaborative index for items.
    """
    # get the items in training data and all data.
    all_items = set()
    train_items = set()
    for user in user_sequence_dict:
        all_items.update(set(user_sequence_dict[user]))
        train_items.update(set(user_sequence_dict[user][:-2]))

    # reindex all training items for calculating the adjacency matrix
    item2id = dict()
    id2item = dict()
    for item in train_items:
        item2id[item] = len(item2id)
        id2item[len(id2item)] = item


    # calculate the co-occurrence of items in the training data as an adjacency matrix
    if float32 > 0:
        adj_matrix = np.zeros((len(item2id), len(item2id)), dtype=np.float32)
    else:
        adj_matrix = np.zeros((len(item2id), len(item2id)))
    for user in user_sequence_dict:
        interactions = user_sequence_dict[user][:-2]
        for pairs in combinations(interactions, 2):
            adj_matrix[item2id[pairs[0]]][item2id[pairs[1]]] += 1
            adj_matrix[item2id[pairs[1]]][item2id[pairs[0]]] += 1


    # get the clustering results for the first layer
    clustering = SpectralClustering(
        n_clusters=cluster_num,
        assign_labels="cluster_qr",
        random_state=0,
        affinity="precomputed",
    ).fit(adj_matrix)
    labels = clustering.labels_.tolist()

    # count the clustering results
    grouping = defaultdict(list)
    for i in range(len(labels)):
        grouping[labels[i]].append((id2item[i],i))

    item_map = dict()
    index_now = 0

    # add current clustering information into the item indexing results.
    item_map, index_now = add_token_to_indexing(item_map, grouping, index_now, token_size)

    # add current clustering info into a queue for BFS
    queue = []
    for group in grouping:
        queue.append(grouping[group])

    # apply BFS to further use spectral clustering for large groups (> token_size)
    while queue:
        group_items = queue.pop(0)

        # if current group is small enough, add the last token to item indexing
        if len(group_items) <= token_size:
            item_list = [items[0] for items in group_items]
            if last_token == 'sequential':
                item_map = add_last_token_to_indexing_sequential(item_map, item_list, token_size)
            elif last_token == 'random':
                item_map = add_last_token_to_indexing_random(item_map, item_list, token_size)
        else:
            # calculate the adjacency matrix for current group
            if float32 > 0:
                sub_adj_matrix = np.zeros((len(group_items), len(group_items)), dtype=np.float32)
            else:
                sub_adj_matrix = np.zeros((len(group_items), len(group_items)))
            for i in range(len(group_items)):
                for j in range(i+1, len(group_items)):
                    sub_adj_matrix[i][j] = adj_matrix[group_items[i][1]][group_items[j][1]]
                    sub_adj_matrix[j][i] = adj_matrix[group_items[j][1]][group_items[i][1]]

            # get the clustering results for current group
            clustering = SpectralClustering(
                n_clusters=cluster_num,
                assign_labels="cluster_qr",
                random_state=0,
                affinity="precomputed",
            ).fit(sub_adj_matrix)
            labels = clustering.labels_.tolist()

            # count current clustering results
            grouping = defaultdict(list)
            for i in range(len(labels)):
                grouping[labels[i]].append(group_items[i])

            # add current clustering information into the item indexing results.
            item_map, index_now = add_token_to_indexing(item_map, grouping, index_now, token_size)

            # push current clustering info into the queue
            for group in grouping:
                queue.append(grouping[group])

    # if some items are not in the training data, assign an index for them
    remaining_items = list(all_items - train_items)
    if len(remaining_items) > 0:
        if last_token == 'sequential':
            item_map = add_last_token_to_indexing_sequential(item_map, remaining_items, token_size)
        elif last_token == 'random':
            item_map = add_last_token_to_indexing_random(item_map, remaining_items, token_size)

    return item_map



def add_token_to_indexing(item_map, grouping, index_now, token_size):
    for group in grouping:
        index_now = index_now % token_size
        for (item, idx) in grouping[group]:
            if item not in item_map:
                item_map[item] = ''
            item_map[item] += f'<CI{index_now}>'
        index_now += 1
    return item_map, index_now

def add_last_token_to_indexing_random(item_map, item_list, token_size):
    last_tokens = random.sample([i for i in range(token_size)], len(item_list))
    for i in range(len(item_list)):
        item = item_list[i]
        if item not in item_map:
            item_map[item] = ''
        item_map[item] += f'<CI{last_tokens[i]}>'
    return item_map

def add_last_token_to_indexing_sequential(item_map, item_list, token_size):
    for i in range(len(item_list)):
        item = item_list[i]
        if item not in item_map:
            item_map[item] = ''
        item_map[item] += f'<CI{i}>'
    return item_map


def get_dict_from_lines(lines):
    """
    Used to get user or item map from lines loaded from txt file.
    """
    index_map = dict()
    for line in lines:
        info = line.split(" ")
        index_map[info[0]] = info[1]
    return index_map




def generate_user_map(user_sequence_dict):
    """
    generate user map based on user sequence dict.
    """
    user_map = dict()
    for user in user_sequence_dict.keys():
        user_map[user] = str(len(user_map) + 1)
    return user_map


def reindex(user_sequence_dict, user_map, item_map):
    """
    reindex the given user sequence dict by given user map and item map
    """
    reindex_user_sequence_dict = dict()
    for user in user_sequence_dict:
        uid = user_map[user]
        items = user_sequence_dict[user]
        reindex_user_sequence_dict[uid] = [item_map[i] for i in items]

    return reindex_user_sequence_dict


def construct_user_sequence_dict(user_sequence):
    """
    Convert a list of string to a user sequence dict. user as key, item list as value.
    """

    user_seq_dict = dict()
    for line in user_sequence:
        user_seq = line.split(" ")
        user_seq_dict[user_seq[0]] = user_seq[1:]
    return user_seq_dict

In [ ]:
# from https://github.com/agiresearch/OpenP5/blob/main/src/utils/generation_trie.py
class Trie(object):
    def __init__(self, sequences: List[List[int]] = []):
        self.trie_dict = {}
        self.len = 0
        if sequences:
            for sequence in sequences:
                Trie._add_to_trie(sequence, self.trie_dict)
                self.len += 1

        self.append_trie = None
        self.bos_token_id = None

    def append(self, trie, bos_token_id):
        self.append_trie = trie
        self.bos_token_id = bos_token_id

    def add(self, sequence: List[int]):
        Trie._add_to_trie(sequence, self.trie_dict)
        self.len += 1

    def get(self, prefix_sequence: List[int]):
        return Trie._get_from_trie(
            prefix_sequence, self.trie_dict, self.append_trie, self.bos_token_id
        )

    @staticmethod
    def load_from_dict(trie_dict):
        trie = Trie()
        trie.trie_dict = trie_dict
        trie.len = sum(1 for _ in trie)
        return trie

    @staticmethod
    def _add_to_trie(sequence: List[int], trie_dict: Dict):
        if sequence:
            if sequence[0] not in trie_dict:
                trie_dict[sequence[0]] = {}
            Trie._add_to_trie(sequence[1:], trie_dict[sequence[0]])

    @staticmethod
    def _get_from_trie(
        prefix_sequence: List[int],
        trie_dict: Dict,
        append_trie=None,
        bos_token_id: int = None,
    ):
        if len(prefix_sequence) == 0:
            output = list(trie_dict.keys())
            if append_trie and bos_token_id in output:
                output.remove(bos_token_id)
                output += list(append_trie.trie_dict.keys())
            return output
        elif prefix_sequence[0] in trie_dict:
            return Trie._get_from_trie(
                prefix_sequence[1:],
                trie_dict[prefix_sequence[0]],
                append_trie,
                bos_token_id,
            )
        else:
            if append_trie:
                return append_trie.get(prefix_sequence)
            else:
                return []

    def __iter__(self):
        def _traverse(prefix_sequence, trie_dict):
            if trie_dict:
                for next_token in trie_dict:
                    yield from _traverse(
                        prefix_sequence + [next_token], trie_dict[next_token]
                    )
            else:
                yield prefix_sequence

        return _traverse([], self.trie_dict)

    def __len__(self):
        return self.len

    def __getitem__(self, value):
        return self.get(value)


def prefix_allowed_tokens_fn(candidate_trie):
    def prefix_allowed_tokens(batch_id, sentence):
        sentence = sentence.tolist()
        trie_out = candidate_trie.get(sentence)
        return trie_out

    return prefix_allowed_tokens

def exact_match(predictions, targets, k):
    batched_predictions = []
    batch_length = len(targets)
    for b in range(batch_length):
        batched_predictions.append(predictions[b * k : (b + 1) * k])
    correct = 0
    for p, t in zip(batched_predictions, targets):
        if t in p:
            correct += 1

    return correct

In [ ]:
def inference(
    seed: int = 2023,
    log_dir: str = "../log",
    logging_level: int = logging.INFO,
    checkpoint_path: str = "../checkpoint",
    backbone: str = 't5-small',
    lora: int = 0,
    data_path: str = "../data",
    item_indexing: str = "sequential",
    tasks: str = 'sequential,straightforward',
    dataset: str = "Beauty",
    cutoff: int = 1024,
    sequential_order: str = "original",
    collaborative_token_size: int = 200,
    collaborative_cluster: int = 20,
    collaborative_last_token: str = "sequential",
    collaborative_float32: int = 0,
    valid_prompt: str = 'seen:0',
    test_prompt: str = 'seen:0',
    metrics: str = 'hit@5,hit@10,ndcg@5,ndcg@10',
    eval_batch_size: int = 32
):
    # setup
    log_file = os.path.join(log_dir, dataset, checkpoint_path.replace('.','').replace('/', '_') + '.log')

    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    logging.basicConfig(filename=log_file, level=logging_level, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))

    set_seed(seed)

    # get all items
    if item_indexing == 'sequential':
        item_index_file = os.path.join(data_path, dataset, f'item_sequential_indexing_{sequential_order}.txt')

    elif item_indexing == 'random':

        item_index_file = os.path.join(data_path, dataset, 'item_random_indexing.txt')

    elif item_indexing == 'collaborative':
        item_index_file = os.path.join(data_path, dataset, f'item_collaborative_indexing_{collaborative_token_size}_{collaborative_cluster}_{collaborative_last_token}.txt')
    else:
        raise NotImplementedError

    item_info = ReadLineFromFile(item_index_file)
    item_map = get_dict_from_lines(item_info)
    all_items = list(item_map.values())

    # load checkpoint
    if 't5' in backbone.lower():
        model = T5ForConditionalGeneration.from_pretrained(checkpoint_path)
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    elif 'llama' in backbone.lower():
        if lora > 0:
            model = LlamaForCausalLM.from_pretrained(
                'meta-llama/' + backbone,
                load_in_8bit=True,
                torch_dtype=torch.float16
            )
            model = PeftModel.from_pretrained(
                model,
                checkpoint_path,
                torch_dtype=torch.float16
            )
        else:
            model = LlamaForCausalLM.from_pretrained(
                checkpoint_path,
                load_in_8bit=True,
                torch_dtype=torch.float16
            )
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
    else:
        raise NotImplementError

    tokenizer.pad_token_id = (
        0  # unk. we want this to be different from the eos token
    )
    tokenizer.padding_side = "left"

    # load test data
    test_data_file = os.path.join(data_path, dataset, f'{dataset}_{tasks}_{item_indexing}_test_{test_prompt}.json')
    test_data = load_dataset("json", data_files=test_data_file, field='data')


    model.eval()

    candidates = all_items
    candidate_trie = Trie(
        [
            [0] + tokenizer.encode(f"{dataset} item_{candidate}")
            for candidate in candidates
        ]
    )
    prefix_allowed_tokens = prefix_allowed_tokens_fn(candidate_trie)

    task_list = np.unique(test_data['train']['task'])
    metrics = metrics.split(',')
    generate_num = max([int(m.split('@')[1]) for m in metrics])
    for task in task_list:
        logging.info(f'testing on {task}')
        subset_data = test_data.filter(lambda example: example['task'] == task)
        dataset = EvaluationDataset(subset_data['train'], tokenizer, cutoff)
        dataloader = DataLoader(dataset, batch_size=eval_batch_size, shuffle=False)
        test_single_task(model, tokenizer, dataloader, metrics, generate_num, prefix_allowed_tokens)



In [ ]:
inference(
    log_dir="/content/drive/MyDrive/OpenP5/logs/",
    checkpoint_path="/content/drive/MyDrive/OpenP5/models/Beauty/sequential/t5-small",
    item_indexing="sequential",
    tasks="sequential,straightforward",
    dataset="Beauty",
    data_path = '/content/drive/MyDrive/OpenP5/data/',
    backbone="t5-small",
    cutoff=1024
)

testing on Beautysequential


100%|██████████| 699/699 [09:05<00:00,  1.28it/s]

hit@5: 0.0017439520636766087
hit@10: 0.0028171533336314447
ndcg@5: 0.001022543131037156
ndcg@10: 0.0013706980899835886
testing on Beautystraightforward


Filter:   0%|          | 0/44726 [00:00<?, ? examples/s]

100%|██████████| 699/699 [08:31<00:00,  1.37it/s]

hit@5: 0.000983767830791933
hit@10: 0.001520368465769351
ndcg@5: 0.0006842352817510766
ndcg@10: 0.0008536550480191407
